# Example: Gridding and Animating Radio Interferometer Measurements

## Introduction

In this notebook we demonstrate the gridding of radio interferometer data using `pyvisgrid` with the intent
to create a time-series animation of the observation.
In this demonstration we will use measured data from the DSHARP measurement of the protoplanetary disk *DoAr25*.

The cleaned fiducial image of this measurement looks like this:

![DoAr25_continuum](./images/DoAr25_continuum.png)


<div class="alert alert-block alert-warning">
<b>Warning:</b> This notebook explains the steps very detailed to enable you to understand the parameters and the concept. If you are only interested in the full script to create the animation (without any preliminary considerations like previews etc.), you can skip to the end of the file.
</div>

## Prerequisites

- Version of [`pyvisgrid`](https://github.com/radionets-project/pyvisgrid) with the optional dependency the `animations` installed.
  To install the optional dependencies for the animation submodule, use:

  ```$ pip install -e ".[animations]"```

- The following shell commands should be available:

  - `wget` OR `curl`
  - `gzip`
  - `tar`
  

In [ ]:
# Import everything we will need

import subprocess
from datetime import timedelta
from pathlib import Path

from pyvisgrid import Gridder
from pyvisgrid.plotting import (
    animate_observation,
    plot_observation_state,
)

First, we will download and decompress the downloaded measurement set.

## Downloading and Decompressing the Data

In [ ]:
url = "https://almascience.eso.org/almadata/lp/DSHARP/MSfiles/DoAr25_continuum.ms.tgz"

archive = Path(f"./data/{url.split('/')[-1]}")
tar = archive.with_suffix(".tar")
ms = archive.with_suffix("")

archive.parent.mkdir(exist_ok=True)

dl_opts = dict(
    check=True, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
if not archive.exists() and not tar.exists() and not ms.exists():
    print(f"Downloading file to {archive} ... This could take a moment ...")
    try:
        subprocess.run(f"wget -O {archive} {url}", **dl_opts)
    except subprocess.CalledProcessError:
        try:
            subprocess.run(f"curl {url} > {archive}", **dl_opts)
        except subprocess.CalledProcessError:
            raise RuntimeError(
                "Neither wget nor curl are installed. Check your installation or "
                "download the file manually and put it into the `data` directory."
            )

else:
    print(f"File {archive} already existent ... Skipping download ...")

if not tar.exists() and not ms.exists():
    print(f"Decompressing {archive} ...")
    subprocess.run(f"gzip -d {archive}", shell=True)

if tar.exists() and not ms.exists():
    print(f"Untaring {tar} ...")
    subprocess.run(f"tar -xf {tar} -C {tar.parent}", shell=True)
    tar.unlink()

## Creating the `Gridder`

After downloading and decompressing the data, we can proceed to initializing the `Gridder` using the `Gridder.from_ms` method.
This method extracts the data from the **M**easurement **S**et and prepares it for the gridding process.

We will be gridding the Stokes `I` component of the observation. 

For creating the gridder, we have to decide for values for the following parameters:

### Mandatory Arguments

1. `fov`: This is the physical width/length of the image in arcseconds. We will be using `fov=9.0` for this example but you are free to change the Field of View.
2. `image_size`: This is an integer value which determines the length **and** with of the resulting image in pixels. We will use `img_size=1000` for our example. Reducing this parameter will lead to a more "blocky" looking image. It can be sensible to change this parameter to reduce the performance impact.

### Optional Arguments

3. *Descriptor ID* `desc_id`. The measurement we are using is composed of smaller measurements which can be switched between by changing this `desc_id`. It is also possible to include all sub-measurements by setting this to `None`. We will be using `desc_id=4` in our example to reduce the performance impact and to avoid visual overload.

In [ ]:
time_diffs = []
for i in range(16):
    gridder = Gridder.from_ms(path=ms, fov=9.0, img_size=500, desc_id=i)
    time_diffs.append(gridder.times.max() - gridder.times.min())

In [ ]:
gridder = Gridder.from_ms(path=ms, fov=9.0, img_size=500, desc_id=None)

In [ ]:
gridder.times.max() - gridder.times.min()

In [ ]:
import numpy as np

np.max(time_diffs)

In [ ]:
len(gridder.times) // gridder.frequencies.size

If you want to preview the gridding results for your specific settings, you can use the code in the following cell.

<div class="alert alert-block alert-info">
<b>Tip:</b> This can be especially helpful when you want to determine something like the crop for the gridded visibilities. The crop determines which part of the image is visible in a plot. In our case for desc_id=4 the gridded visibilities are only visible in the center of the image and are thus not really visible. Therfore we only plot a smaller part of the image. 
</div>

In [ ]:
_ = gridder.grid()
gridder.plot_ungridded_uv(show_times=False, marker_size=1)
gridder.plot_mask(norm="log", crop=[(200, 300), (200, 300)])
gridder.plot_dirty_image()
None

## Gridding the Time Series

To find out how many grid points we have in our dataset, we can use the built-in `len` function on the `Gridder` instance.

In [ ]:
print(
    f"Number of total visibilities: {len(gridder)} for {len(gridder.frequencies)} frequency bands "
    f"--> {len(gridder) // len(gridder.frequencies)} per frequency band"
)

According to that number we can now choose our step size for the time series, meaning how many time visibilities will be added per gridded time step. The number of resulting time steps is then given by `N_visibilities // time_steps`. These time steps will be the frames of the animation.

An animation with 30 fps (*frames per second*) will then have a length of `(N_visibilities // time_steps) / fps = (N_visibilities // time_steps) / 30`

We will use `step_size=100` by default, but feel free to change it if you want a shorter animation.
You can also change the frame rate if you want. We will use `fps=30` by default

<div class="alert alert-block alert-info">
<b>Tip:</b> If you make `step_size` too large, consider reducing the frame rate to avoid very short animations.
</div>



In [ ]:
step_size = 250
fps = 30

In [ ]:
frames = len(gridder) // step_size
animation_length = frames / fps

# We will use the timedelta to format the time in the format hh:mm:ss
print(
    f"Animation length @ {fps} fps for a step size of {step_size}: {timedelta(seconds=animation_length)}"
)

We can also choose so called `time_start_idx` and `time_end_idx`. If you don't want to have a time series which covers the whole observation you can change these to start or stop your time series at a specific index. For our example, we won't use these and thus create a time series of the entire observation.

In [ ]:
grid_data_series = gridder.grid_time_series(step_size=step_size)

## Configuring and Previewing the Animation

Now we can finally create our animation.

For that we should first consider which elements should be present in our plot and in which constellation.
We can include the following elements in our plot:

- A model of the **earth with the projected source position and the array positions** on it (key: `earth`)
- A plot of the **ungridded $(u,v)$ coordinates** and their corresponding time step (if desired) (key: `uv`)
- A plot of the **dirty image** (key: `di`)
- Plots of the masks (the **gridded visibilities**). Since the visibilities are complex, we have to use two plots to show them. (keys: `mask_hi` and `mask_lo`).

These have to be sorted into a specific order and we do that using a multidimensional array describing the rows and columns of our image.

An example:
```
[["mask_hi", "uv"], 
 ["mask_lo", "di"]]
```

Would mean that we have a 2x2 grid of plots with the masks on the left column (`mask_hi` on top and `mask_lo` below that) and the ungridded $(u,v)$ coordinates above the dirty image in the right column.

If we want one plot to be larger, we can also mention it multiple times:

```
[["earth", "uv"], 
 ["earth", "di"]]
```

This time the left column will entirely consist of the earth layout and to the right of this we can see the ungridded $(u,v)$ coordinates above the dirty image.

By default the following layout will be used:

```
[["mask_hi", "earth", "uv"], 
 ["mask_lo", "earth", "di"]]
```

We will provide our layout choice to the method via the plot_positions keyword.

We can use the `pyvisgrid.plotting.plot_observation_state` method to display the plot constellation for the an arbitrary observation timestep from our `GridDataSeries`.

In [ ]:
# In this case we have to decompose the data saved in the
# `GridDataSeries` object to provide it to the method.
vis_data, u, v, times = grid_data_series[-1]

plot_observation_state(
    gridder=gridder,
    vis_data=vis_data,
    u=u,
    v=v,
    times=times,
    plot_positions=[["mask_hi", "earth", "uv"], ["mask_lo", "earth", "di"]],
    mask_crop=[(200, 300), (200, 300)],
)
None

Above you can see the default constellation. You might notice that none of the plots have visible ticks on their $x$ and $y$ axes and only some have visible ticks on their colorbar. If you want to change this, the `axes_options` are what you want to change.

This essentially is a large dictionary containing the parameters which are provided to the e.g. the matplotlib methods or control what what will get displayed in the image.

This for example is the `axes_options` section which is related to the ungridded $(u,v)$ plot:

```
"uv": {
    "show_title": True,
    "title_fontsize": "medium",
    "axes_ticks": False,
    "axes_labels": True,
    "axes_fontsize": "x-small",
    "show_times": True,
    "cmap": "magma",
    "show_cbar": True,
    "cbar_ticks": True,
    "cbar_label": True,
    "cbar_fontsize": "small",
    "color": _default_colors[4],
    "aspect": "equal",
},
```

The good thing is: If you want to change only one parameter in the options, you don't have to provide the entire settings dictionary again but you can just change the parameter and it will be overridden in the configuration for this run.

Say you don't want the relative time since the beginning of the observations to be displayed as a colorbar. Than you can just set `axes_options={"uv": {"show_times": False}}` as a argument in the function.

In [ ]:
plot_observation_state(
    gridder=gridder,
    vis_data=vis_data,
    u=u,
    v=v,
    times=times,
    plot_positions=[["mask_hi", "earth", "uv"], ["mask_lo", "earth", "di"]],
    mask_crop=[(200, 300), (200, 300)],
    axes_options={"uv": {"show_times": False}},
)
None

If you want to show the real and imaginary parts instead of the amplitude and phase, you can change the `mask_mode` from `"amp_phase"` to `"real_imag"`.

If for some reason you want to change which component is used for `mask_hi` and which for `mask_lo` (default: `amp` & `real` -> `mask_hi` / `phase` & `imag` -> `mask_lo`), you can set `swap_masks=True`.

In [ ]:
plot_observation_state(
    gridder=gridder,
    vis_data=vis_data,
    u=u,
    v=v,
    times=times,
    mask_mode="real_imag",
    plot_positions=[["mask_hi", "earth", "uv"], ["mask_lo", "earth", "di"]],
    mask_crop=[(200, 300), (200, 300)],
    axes_options={"uv": {"show_times": False}},
)
None

## Creating and Saving the Animation

Now that we know how we can configure what we want to plot, we can finally animate it. This is probably the shortest steps of all, since it is a single function call to `pyvisgrid.plotting.animate_observation`.

The only thing we have to provide is the output file in the `save_to` parameter.
In our case, we want to save an MP4 file to a directory we call `build`.

To control the resolution of the animation, we can change the `dpi` argument. Here we use `dpi=600`.

Watch out: Since we pass the entire `GridDataSeries` to the animation, we don't have to decompose any outputs but can just provide `Gridder` and `GridDataSeries` to the function.

In [ ]:
Path("./build").mkdir(exist_ok=True)

animate_observation(
    gridder=gridder,
    series=grid_data_series,
    mask_mode="amp_phase",
    mask_crop=[(200, 300), (200, 300)],
    plot_positions=[["mask_hi", "earth", "uv"], ["mask_lo", "earth", "di"]],
    fps=fps,
    save_to="./build/animation.mp4",
    dpi=600,
)

## Full Script

In [ ]:
import subprocess
from datetime import timedelta
from pathlib import Path

from pyvisgrid import Gridder
from pyvisgrid.plotting import (
    animate_observation,
    plot_observation_state,
)

# Download and Decompress Data

url = "https://almascience.eso.org/almadata/lp/DSHARP/MSfiles/DoAr25_continuum.ms.tgz"

archive = Path(f"./data/{url.split('/')[-1]}")
tar = archive.with_suffix(".tar")
ms = archive.with_suffix("")

archive.parent.mkdir(exist_ok=True)

dl_opts = dict(
    check=True, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
if not archive.exists() and not tar.exists() and not ms.exists():
    print(f"Downloading file to {archive} ... This could take a moment ...")
    try:
        subprocess.run(f"wget -O {archive} {url}", **dl_opts)
    except subprocess.CalledProcessError:
        try:
            subprocess.run(f"curl {url} > {archive}", **dl_opts)
        except subprocess.CalledProcessError:
            raise RuntimeError(
                "Neither wget nor curl are installed. Check your installation or "
                "download the file manually and put it into the `data` directory."
            )

else:
    print(f"File {archive} already existent ... Skipping download ...")

if not tar.exists() and not ms.exists():
    print(f"Decompressing {archive} ...")
    subprocess.run(f"gzip -d {archive}", shell=True)

if tar.exists() and not ms.exists():
    print(f"Untaring {tar} ...")
    subprocess.run(f"tar -xf {tar} -C {tar.parent}", shell=True)
    tar.unlink()

# Creating Gridder and Animation

gridder = Gridder.from_ms(path=ms, fov=9.0, img_size=500, desc_id=4)

step_size = 500
fps = 30
frames = len(gridder) // step_size

grid_data_series = gridder.grid_time_series(step_size=step_size)

Path("./build").mkdir(exist_ok=True)

animate_observation(
    gridder=gridder,
    series=grid_data_series,
    mask_mode="amp_phase",
    plot_positions=[["mask_hi", "earth", "uv"], ["mask_lo", "earth", "di"]],
    fps=fps,
    save_to="./build/animation.mp4",
    dpi=600,
)